# Budget-matched baselines on SIB-200 Yoruba

`exp_xlmr_lr_sweep.ipynb` found that XLM-R's failure on SIB-200 was a **step budget** problem, not
a Yoruba problem: 2 of 25 seeds cleared the uniform-random bar at 352 steps, against 18 of 25 at
1056. Its best configuration is lr 1e-5 at 1056 steps, 5/5 seeds converged, mean 0.408.

**mmBERT's 0.529 was measured at 352 steps**, so setting it against 0.408 compares two different
budgets — which is exactly what a fixed step budget exists to prevent. This notebook measures the
baselines at a budget where both models actually train, so the comparison means something.

Two design points, because neither is free:

- **Symmetry.** XLM-R's 0.408 is the best of five learning rates. Giving mmBERT a single learning
  rate at the new budget would hand XLM-R a best-of-five advantage and call the difference a model
  property. mmBERT gets the same five.
- **The control moves too.** The random-init floor of 0.107 is also a 352-step number. A control
  measured at a budget nobody else is using has stopped being a control.

**Cost on an A100** — roughly 96 s/seed at 1056 steps and 32 s at 352:

| block | runs | time |
|---|---|---|
| mmBERT, 5 lrs × 1056 steps × 5 seeds | 25 | ~40 min |
| mmBERT, 2e-5 × 352 steps, topped up to 5 seeds | 5 | ~3 min |
| random init × 1056 steps × 5 seeds | 5 | ~5 min, plus corpus prep |

`QUICK = True` cuts the first block to lr 2e-5 alone (~8 min) if you are short of runtime.

In [8]:
import os, sys
REPO = '/content/WashingtonCsed504'
FORK = 'https://github.com/patlkwok/WashingtonCsed504.git'   # YOUR fork, not upstream
if not os.path.exists(REPO):
    !git clone -q {FORK} {REPO}
sys.path.insert(0, f'{REPO}/src/a2-nlp')
import session; factory = session.start(prepare=False)   # corpus only needed for the control

python 3.12.13 | NVIDIA A100-SXM4-80GB (85 GB, sm_80) | bf16: True
  packages already present
ready — cwd /content/WashingtonCsed504/src/a2-nlp


In [2]:
import importlib, time
import numpy as np
import ft_api as ft
importlib.reload(ft)

GPU = ft.gpu_name()
print('ft_api', ft.API_VERSION, '| GPU', GPU)
if 'A100' not in GPU and 'PRO 6000' not in GPU:
    print('  NOTE: the time estimates assume an A100. This is not one; expect them to differ.')

sib = ft.load_sib200('yor_Latn')

# Same bar as the sweep notebook, measured rather than typed in, so the two agree by
# construction instead of by a constant copied between them.
gold, k = sib['test']['label'], len(sib['labels'])
rng = np.random.default_rng(0)
UNIFORM = float(np.mean([ft.macro_f1(gold, rng.integers(0, k, len(gold)).tolist(), k)
                         for _ in range(400)]))
print(f'uniform-random bar {UNIFORM:.3f} on {len(gold)} test items')

ft_api (1, 3) | GPU NVIDIA A100-SXM4-80GB


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/47.9k [00:00<?, ?B/s]

train.tsv:   0%|          | 0.00/128k [00:00<?, ?B/s]

dev.tsv:   0%|          | 0.00/17.0k [00:00<?, ?B/s]

test.tsv:   0%|          | 0.00/37.6k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

sib200/yor_Latn: train 701 validation 99 test 204 | 7 classes (chance 0.143) | 1.1% decomposed -> NFC
uniform-random bar 0.135 on 204 test items


## The runs

**This overwrites mmBERT's canonical 352-step row.** The record tag does not carry the seed count,
so topping that cell up from 3 seeds to 5 rewrites it in place — the same thing the LR sweep did to
XLM-R's row. It is better evidence at the same setting, the record states its own seed list, and
the 3-seed version stays in git history. Set `TOPUP_352 = False` to leave it alone.

In [3]:
MMBERT = 'jhu-clsp/mmBERT-base'
LRS    = [5e-6, 1e-5, 2e-5, 3e-5, 5e-5]
SEEDS  = (0, 1, 2, 3, 4)
REUSE  = True

QUICK        = False   # True -> only lr 2e-5 at 1056 steps
TOPUP_352    = True    # 5 seeds on mmBERT's existing 352-step cell
RUN_CONTROL  = True    # random-init floor at 1056; prepares the Yoruba corpus first

t0 = time.time()

for lr in ([2e-5] if QUICK else LRS):
    ft.evaluate(MMBERT, task='sib200', lr=lr, steps=1056, seeds=SEEDS,
                data=sib, reuse=REUSE, label=f'mmBERT lr{lr:g} st1056')

if TOPUP_352:
    ft.evaluate(MMBERT, task='sib200', lr=2e-5, steps=352, seeds=SEEDS,
                data=sib, reuse=REUSE, label='mmBERT base')

print(f'\nbaselines done in {(time.time() - t0) / 60:.1f} min on {GPU}')

config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/46.4k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: jhu-clsp/mmBERT-base
Key               | Status     | 
------------------+------------+-
decoder.weight    | UNEXPECTED | 
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: jhu-clsp/mmBERT-base
Key               | Status     | 
------------------+------------+-
decoder.weight    | UNEXPECTED | 
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  mmBERT base                macro_f1 0.521 +/-0.020 (seed sd) | 95% CI [0.455, 0.573] | n_train 701 | 41s/seed

baselines done in 57.7 min on NVIDIA A100-SXM4-80GB


In [4]:
if RUN_CONTROL:
    factory = session.start(corpus='yor')      # verifies tokenizer fingerprint 15abd33de5af
    rand = factory.random_init('yor')
    ft.evaluate(rand, task='sib200', lr=ft.FT_LR_SCRATCH, steps=1056, seeds=SEEDS,
                data=sib, reuse=REUSE, label='random init')
else:
    print('skipped — the 1056-step floor will be missing from the table below')

python 3.12.13 | NVIDIA A100-SXM4-80GB (85 GB, sm_80) | bf16: True
  packages already present
yor: preparing yor_Latn
  using the shared tokenizer at 'tokenizers/yor-bpe16k' (not training a new one)


README.md:   0%|          | 0.00/329k [00:00<?, ?B/s]

    [fineweb2] 79,999 docs / 260M chars in 64s                        
    encoded 79,999 docs -> 69,596,452 tokens in 69s                        

  decoded sample: '<s> Ẹ̀gbá\nÀwọn àkóónú\nILẸ̀ Ẹ̀GBÁ[àtúnṣe | edit source]\nÓ se pàtàkì láti mọ díẹ̀ nípa ìtàn ilẹ̀ Ẹ̀gbá àti irú ènìyàn tí ń gbé ìlú Ẹ̀gbá. Ìdí èyí ni pé yóò jẹ́'
  69,096,452 train + 500,000 val tokens, 3.73 chars/token
  vocabulary fingerprint 15abd33de5af -- runs only compare across matching fingerprints

  corpus 'yor' ready, tokenizer 15abd33de5af
ready — cwd /content/WashingtonCsed504/src/a2-nlp


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/133 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/133 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: /content/WashingtonCsed504/src/a2-nlp/runs/yor_random_init
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/133 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: /content/WashingtonCsed504/src/a2-nlp/runs/yor_random_init
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  random init                macro_f1 0.403 +/-0.047 (seed sd) | 95% CI [0.351, 0.446] | n_train 701 | 30s/seed


## Every 5-seed cell, grouped by budget

Cells are never compared across the horizontal rule. `conv` is the number of seeds clearing the
uniform-random bar; a cell that is not 5/5 is not a single number, whatever its mean says.

In [9]:
MODELS = [('mmBERT', 'mmBERT-base'), ('XLM-R', 'xlm-roberta-base'),
          ('random init', 'yor-random-init')]

def cells(slug, steps):
    return sorted((r for r in ft.results(task='sib200')
                   if r['model_slug'] == slug and r['steps'] == steps
                   and len(r['seeds']) == len(SEEDS)),
                  key=lambda r: -r['mean'])

def conv(r):
    return sum(x > UNIFORM for x in r['scores'])

# A model with no records at all used to vanish from these tables in silence, which is how an
# earlier run printed a head-to-head missing an entire model and said nothing. Records reach a
# fresh runtime only by being committed and pulled, so absence almost always means "not pushed"
# rather than "not measured" -- and the two look identical once the row is simply gone.
absent = [label for label, slug in MODELS
          if not any(cells(slug, s) for s in (352, 1056))]
if absent:
    print('!' * 66)
    print(f'NO {len(SEEDS)}-SEED RECORDS FOR: {", ".join(absent)}')
    print('The tables below are INCOMPLETE. A fresh runtime gets runs/ by cloning the fork,')
    print('so commit and push the records, then re-run session.start() to pull them in.')
    print('!' * 66)

for steps in (352, 1056):
    print(f'\n{"=" * 62}\n{steps} steps\n{"=" * 62}')
    print(f'{"model":<13}{"lr":>8}{"conv":>7}{"mean":>8}{"sd":>7}   95% CI')
    for label, slug in MODELS:
        rows = cells(slug, steps)
        if not rows:
            print(f'{label:<13}{"-- no records --":>38}')
            continue
        for r in rows:
            print(f'{label:<13}{r["lr"]:>8.0e}{f"{conv(r)}/{len(SEEDS)}":>7}'
                  f'{r["mean"]:>8.3f}{r["sd"]:>7.3f}   [{r["ci"][0]:.3f}, {r["ci"][1]:.3f}]')


352 steps
model              lr   conv    mean     sd   95% CI
mmBERT          2e-05    5/5   0.521  0.020   [0.455, 0.573]
XLM-R           3e-05    1/5   0.139  0.081   [0.122, 0.155]
XLM-R           2e-05    1/5   0.110  0.068   [0.094, 0.125]
XLM-R           5e-06    0/5   0.099  0.015   [0.084, 0.113]
XLM-R           1e-05    0/5   0.089  0.026   [0.074, 0.102]
XLM-R           5e-05    0/5   0.057  0.000   [0.046, 0.067]
random init                        -- no records --

1056 steps
model              lr   conv    mean     sd   95% CI
mmBERT          5e-05    5/5   0.574  0.035   [0.501, 0.627]
mmBERT          2e-05    5/5   0.556  0.016   [0.490, 0.599]
mmBERT          3e-05    5/5   0.548  0.011   [0.484, 0.595]
mmBERT          1e-05    5/5   0.499  0.028   [0.435, 0.548]
mmBERT          5e-06    5/5   0.483  0.034   [0.422, 0.529]
XLM-R           1e-05    5/5   0.408  0.037   [0.351, 0.455]
XLM-R           2e-05    4/5   0.394  0.140   [0.345, 0.434]
XLM-R           3e-05    4

## The head-to-head

Each model is represented by its **best fully-converged cell** at that budget. That is a choice
made on the test set — SIB-200 ships a 99-item validation split, so select there instead before
quoting a single number in a writeup. It is reported here because the alternative, averaging over
configurations that never trained, is worse.

Every difference passes two gates: it must exceed **0.06 macro-F1**, which is what 204 test items
resolve, and the pooled bootstrap intervals must not overlap.

**The control comes first.** A model level with a randomly initialised encoder has contributed
nothing, however far above chance it sits — at a long enough budget the classifier head alone
learns a good deal from 701 labels, and the floor moves up with it. Two models beating each other
only means something once both have beaten the floor.

Where a model has no cell to show, the reason is printed: *no records* and *records but none
converged* are different problems and only one of them is about the model.

In [10]:
FLOOR = 0.06     # what 204 test items resolve

def best(slug, steps):
    """Best fully-converged cell, and if there is none, which of three reasons applies.

    Not pulled, wrong seed count, and did not converge look identical once the row is simply
    absent -- and only the third is a fact about the model."""
    rows = cells(slug, steps)
    if not rows:
        others = [r for r in ft.results(task='sib200')
                  if r['model_slug'] == slug and r['steps'] == steps]
        if others:
            got = sorted({len(r['seeds']) for r in others})
            return None, f'only {got}-seed record(s) here; this table needs {len(SEEDS)}'
        return None, 'no record at this budget (never run, or not pulled)'
    full = [r for r in rows if conv(r) == len(SEEDS)]
    if not full:
        return None, f'{len(rows)} cell(s) measured, none reaching {len(SEEDS)}/{len(SEEDS)}'
    return full[0], ''

def verdict(a, b, na, nb):
    gap = a['mean'] - b['mean']
    disjoint = a['ci'][0] > b['ci'][1] or b['ci'][0] > a['ci'][1]
    if not disjoint:
        tag = 'NOT separated (intervals overlap)'
    elif abs(gap) < FLOOR:
        tag = f'below the {FLOOR} floor'
    else:
        tag = f'{na if gap > 0 else nb} ahead by {abs(gap):.3f}'
    print(f'    {na:<12}{a["mean"]:.3f}  vs  {nb:<12}{b["mean"]:.3f}   '
          f'gap {gap:+.3f}   {tag}')

for steps in (352, 1056):
    print(f'\n{steps} steps')
    floor, floor_why = best('yor-random-init', steps)
    picked = {}
    for label, slug in (('mmBERT', 'mmBERT-base'), ('XLM-R', 'xlm-roberta-base')):
        r, why = best(slug, steps)
        picked[label] = r
        print(f'  {label:<12}' + (f'{r["mean"]:.3f} at lr {r["lr"]:.0e}' if r else f'-- {why}'))
    print(f'  {"control":<12}' + (f'{floor["mean"]:.3f}' if floor else f'-- {floor_why}'))

    if floor:
        print('  against the untrained control:')
        for label, r in picked.items():
            if r:
                verdict(r, floor, label, 'control')
    else:
        print('  no control at this budget -- nothing below to measure against')

    if picked['mmBERT'] and picked['XLM-R']:
        print('  head to head:')
        verdict(picked['mmBERT'], picked['XLM-R'], 'mmBERT', 'XLM-R')


352 steps
  mmBERT      0.521 at lr 2e-05
  XLM-R       -- 5 cell(s) measured, none reaching 5/5
  control     -- only [3]-seed record(s) here; this table needs 5
  no control at this budget -- nothing below to measure against

1056 steps
  mmBERT      0.574 at lr 5e-05
  XLM-R       0.408 at lr 1e-05
  control     0.403
  against the untrained control:
    mmBERT      0.574  vs  control     0.403   gap +0.170   mmBERT ahead by 0.170
    XLM-R       0.408  vs  control     0.403   gap +0.004   NOT separated (intervals overlap)
  head to head:
    mmBERT      0.574  vs  XLM-R       0.408   gap +0.166   mmBERT ahead by 0.166


## Reading it

The interesting outcome is not which model wins. It is whether the gap **survives giving XLM-R a
budget it can train in** — the whole 352-step comparison was decided by a constant inherited from
an earlier notebook's 8-epoch loop, not chosen for this question.

If mmBERT stays ahead at 1056 steps, the vocabulary argument keeps a working downstream example.
If the gap closes, then the earlier result was about optimisation and not about Yoruba, and the
downstream table should say so.

Either way, quote these numbers from `ft.results()` rather than by hand from this output.

In [7]:
session.save_results()   # asserts Drive is really mounted before copying

Mounted at /content/drive
copied 257 result files -> /content/drive/MyDrive/csed504-runs


257